In [1]:
import pandas as pd
import numpy as np
from pymongo import MongoClient

# Korrigierte Zentren (Lat, Lon)
# Major Atmospheric Gamma Imaging Cherenkov 
SC_PALMA = (40.2820, -3.3339) 
# National Air and Space Museum (Washington DC) - falls das dein Ziel war
NASM_COORDS = (51.2839, 7.220)

def haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Erdradius in km
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    a = np.sin(dphi / 2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2)**2
    c = 2 * np.atan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

# Verbindung
client = MongoClient("mongodb://localhost:27017/")
db = client["Espana"]
col = db["ciudad"]

# Abfrage mit Prüfung auf das Geo-Feld (achte auf den Namen in deiner DB!)
cursor = col.find(
    {"Coordinates.type": "Point"}, 
    {"Ciudad": 1, "Provincia": 1, "Altitud": 1, "Coordinates": 1}
)

data = []
for doc in cursor:
    # GeoJSON Format ist [lon, lat]
    coords = doc.get('Coordinates', {}).get('coordinates')
    if coords and len(coords) == 2:
        lon, lat = coords
        
        dist_palma = round(haversine(lat, lon, SC_PALMA[0], SC_PALMA[1]), 2)
        dist_air_space = round(haversine(lat, lon, NASM_COORDS[0], NASM_COORDS[1]), 2)
        
        data.append({
            "Stadt": doc.get("Ciudad"),
            "Provinz": doc.get("Provincia"),
            "Höhe": doc.get("Altitud"),
            "AeropuertoMad_km": dist_palma,
            "Stoppenberg_km": dist_air_space
        })

if data:
    df = pd.DataFrame(data)
    # Sortierung nach Distanz zu La Palma
    df_sorted = df.sort_values(by="Stoppenberg_km")
    
    print(df_sorted.to_string(index=False))
else:
    print("Keine Dokumente mit 'Coordinates.type: Point' gefunden.")

                      Stadt                Provinz  Höhe  AeropuertoMad_km  Stoppenberg_km
                    Portbou                 Gerona    28            580.00         1051.72
               Fuenterrabía              Guipúzcoa    16            366.73         1109.67
                       Irún              Guipúzcoa    20            359.50         1111.10
                    Santoña              Cantabria     7            351.60         1183.82
                  Barcelona              Barcelona    13            469.80         1184.84
          Torrente de Cinca                 Huesca   109            335.74         1210.69
          Torrente de Cinca                quatsch   109            335.74         1210.69
                      Gijón               Asturias     3            410.54         1293.31
           Miedes de Aragón               Zaragoza   758            189.34         1298.42
                    Perelló              Tarragona   150            319.35         1305.62